# Dimensionality Reduction
**Course:** Foundations of Machine Learning  
**Instructor:** Sayan CHAKI, LIRIS (UMR 5205 CNRS), École Centrale de Lyon, Université Lumière Lyon 2, INSA Lyon

**Lab 4.** Curse of dimensionality, PCA from scratch (eigen and SVD), eigen-digits and reconstruction, denoising, PCA in pipelines, LDA, t-SNE, kernel PCA.

> Run in Google Colab: *Runtime → Run all*. All datasets ship with scikit-learn, so no download is needed.


## 0. Setup

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
np.random.seed(0)
plt.rcParams["figure.figsize"] = (7, 4.5)
plt.rcParams["axes.grid"] = True

from sklearn.decomposition import PCA, KernelPCA
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.manifold import TSNE
from sklearn.linear_model import LogisticRegression
from sklearn.datasets import load_digits, make_circles

## 1. The curse of dimensionality: distance concentration
For uniform random points, the relative gap between the farthest and nearest neighbour shrinks as $d$ grows.

In [ ]:
rng = np.random.default_rng(0)
dims = [1, 2, 5, 10, 50, 100, 500, 1000]
ratios = []
for d in dims:
    P = rng.uniform(size=(500, d))
    q = rng.uniform(size=d)
    dist = np.linalg.norm(P - q, axis=1)
    ratios.append((dist.max() - dist.min()) / dist.min())
plt.loglog(dims, ratios, "o-")
plt.xlabel("dimension d"); plt.ylabel("(max - min) / min distance")
plt.title("Distances concentrate in high dimension"); plt.show()

## 2. PCA from scratch on 2D data
1. Centre the data. 2. Compute the covariance $\boldsymbol\Sigma$. 3. Eigen-decompose. 4. Project.

In [ ]:
mean = [2, 1]
cov = [[3, 1.8], [1.8, 1.5]]
X2 = rng.multivariate_normal(mean, cov, size=300)

mu = X2.mean(0)
Xc = X2 - mu
Sigma = Xc.T @ Xc / (len(X2) - 1)
eigval, eigvec = np.linalg.eigh(Sigma)          # ascending order
order = np.argsort(eigval)[::-1]
eigval, eigvec = eigval[order], eigvec[:, order]
print("eigenvalues:", eigval.round(3))
print("explained variance ratio:", (eigval / eigval.sum()).round(3))

plt.scatter(X2[:, 0], X2[:, 1], s=10, alpha=0.5)
for val, vec, c in zip(eigval, eigvec.T, ["r", "g"]):
    plt.arrow(*mu, *(2 * np.sqrt(val) * vec), color=c, width=0.04, head_width=0.2)
plt.axis("equal"); plt.title("Principal directions (length = 2 std)"); plt.show()

### 2.1 Same result with the SVD and with scikit-learn
Directions are defined up to sign.

In [ ]:
U, S, Vt = np.linalg.svd(Xc, full_matrices=False)
print("SVD eigenvalues   :", (S**2 / (len(X2) - 1)).round(3))
sk = PCA().fit(X2)
print("sklearn variances :", sk.explained_variance_.round(3))
print("|cos| between eigvec and sklearn components:", np.abs(np.sum(eigvec.T * sk.components_, axis=1)).round(6))

### 2.2 Projection and reconstruction onto PC1
Reconstruction error equals the discarded eigenvalue.

In [ ]:
z = Xc @ eigvec[:, 0]
X_rec = mu + np.outer(z, eigvec[:, 0])
plt.scatter(X2[:, 0], X2[:, 1], s=10, alpha=0.4, label="original")
plt.scatter(X_rec[:, 0], X_rec[:, 1], s=10, c="r", label="reconstruction (k=1)")
for i in range(0, 300, 15):
    plt.plot([X2[i, 0], X_rec[i, 0]], [X2[i, 1], X_rec[i, 1]], "k-", lw=0.5)
plt.axis("equal"); plt.legend(); plt.show()
err = np.sum((X2 - X_rec) ** 2) / (len(X2) - 1)
print(f"reconstruction error = {err:.4f}   discarded eigenvalue = {eigval[1]:.4f}")

## 3. A reusable PCA class

In [ ]:
class MyPCA:
    def __init__(self, n_components):
        self.k = n_components
    def fit(self, X):
        self.mean_ = X.mean(0)
        U, S, Vt = np.linalg.svd(X - self.mean_, full_matrices=False)
        var = S**2 / (len(X) - 1)
        self.components_ = Vt[:self.k]
        self.explained_variance_ = var[:self.k]
        self.explained_variance_ratio_ = var[:self.k] / var.sum()
        return self
    def transform(self, X):
        return (X - self.mean_) @ self.components_.T
    def inverse_transform(self, Z):
        return Z @ self.components_ + self.mean_

## 4. Digits: variance, eigen-digits and reconstructions
1797 images of size 8x8 (64 features).

In [ ]:
digits = load_digits()
X, y = digits.data, digits.target
full = MyPCA(64).fit(X)
cum = np.cumsum(full.explained_variance_ratio_)
fig, ax = plt.subplots(1, 2, figsize=(13, 4))
ax[0].bar(range(1, 65), full.explained_variance_ratio_); ax[0].set_title("Scree plot"); ax[0].set_xlabel("component")
ax[1].plot(range(1, 65), cum, "o-", ms=3); ax[1].axhline(0.95, c="r", ls="--")
ax[1].set_title("Cumulative explained variance"); ax[1].set_xlabel("number of components")
plt.show()
print("components for 90% variance:", np.searchsorted(cum, 0.90) + 1)
print("components for 95% variance:", np.searchsorted(cum, 0.95) + 1)

In [ ]:
fig, axes = plt.subplots(2, 8, figsize=(14, 4))
axes[0, 0].imshow(full.mean_.reshape(8, 8), cmap="gray"); axes[0, 0].set_title("mean")
for i, ax in enumerate(axes.ravel()[1:]):
    ax.imshow(full.components_[i].reshape(8, 8), cmap="RdBu"); ax.set_title(f"PC{i+1}")
for ax in axes.ravel(): ax.axis("off")
plt.suptitle("Mean digit and the first 15 'eigen-digits'"); plt.show()

In [ ]:
ks = [2, 5, 10, 20, 40, 64]
fig, axes = plt.subplots(4, len(ks) + 1, figsize=(12, 7))
for r, idx in enumerate([0, 11, 42, 99]):
    axes[r, 0].imshow(X[idx].reshape(8, 8), cmap="gray"); axes[r, 0].set_title("original" if r == 0 else "")
    for c, k in enumerate(ks, start=1):
        p = MyPCA(k).fit(X)
        rec = p.inverse_transform(p.transform(X[idx:idx+1]))
        axes[r, c].imshow(rec.reshape(8, 8), cmap="gray")
        if r == 0: axes[r, c].set_title(f"k={k}")
for ax in axes.ravel(): ax.axis("off")
plt.suptitle("Reconstructions with k components"); plt.show()

### 4.1 PCA for denoising

In [ ]:
noisy = X + rng.normal(0, 4, X.shape)
p = PCA(n_components=0.60).fit(noisy)            # keep 60% of the (noisy) variance
denoised = p.inverse_transform(p.transform(noisy))
print("components kept:", p.n_components_)
fig, axes = plt.subplots(3, 10, figsize=(13, 4))
for i in range(10):
    for r, img in enumerate([X, noisy, denoised]):
        axes[r, i].imshow(img[i].reshape(8, 8), cmap="gray"); axes[r, i].axis("off")
axes[0, 0].set_title("clean", loc="left"); axes[1, 0].set_title("noisy", loc="left"); axes[2, 0].set_title("denoised", loc="left")
plt.show()

## 5. PCA as preprocessing: accuracy vs number of components
PCA is fitted **inside** the pipeline, so each CV fold refits it (no leakage).

In [ ]:
from sklearn.svm import SVC
ks = [2, 5, 10, 15, 20, 30, 40, 64]
res = {"logistic": [], "RBF SVM": []}
for k in ks:
    res["logistic"].append(cross_val_score(make_pipeline(StandardScaler(), PCA(k), LogisticRegression(max_iter=3000)), X, y, cv=5).mean())
    res["RBF SVM"].append(cross_val_score(make_pipeline(StandardScaler(), PCA(k), SVC()), X, y, cv=5).mean())
for name, v in res.items():
    plt.plot(ks, v, "o-", label=name)
plt.xlabel("PCA components"); plt.ylabel("5-fold CV accuracy"); plt.legend(); plt.show()

## 6. 2D visualisation: PCA vs LDA vs t-SNE

In [ ]:
Xs = StandardScaler().fit_transform(X)
Z_pca = PCA(2).fit_transform(Xs)
Z_lda = LinearDiscriminantAnalysis(n_components=2).fit_transform(Xs, y)
Z_tsne = TSNE(n_components=2, perplexity=30, init="pca", random_state=0).fit_transform(Xs)

fig, axes = plt.subplots(1, 3, figsize=(18, 5.5))
for ax, Z, t in zip(axes, [Z_pca, Z_lda, Z_tsne], ["PCA (unsupervised, linear)", "LDA (supervised, linear)", "t-SNE (nonlinear)"]):
    sc = ax.scatter(Z[:, 0], Z[:, 1], c=y, cmap="tab10", s=8)
    for d in range(10):
        cx, cy = np.median(Z[y == d], axis=0)
        ax.text(cx, cy, str(d), fontsize=14, weight="bold", ha="center",
                bbox=dict(facecolor="white", alpha=0.7, lw=0))
    ax.set_title(t); ax.set_xticks([]); ax.set_yticks([])
plt.show()

### 6.1 t-SNE is sensitive to perplexity
Cluster sizes and inter-cluster distances in t-SNE plots are **not** reliable.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
for ax, perp in zip(axes, [5, 30, 100]):
    Z = TSNE(n_components=2, perplexity=perp, init="pca", random_state=0).fit_transform(Xs[:800])
    ax.scatter(Z[:, 0], Z[:, 1], c=y[:800], cmap="tab10", s=8); ax.set_title(f"perplexity={perp}")
    ax.set_xticks([]); ax.set_yticks([])
plt.show()

## 7. Kernel PCA on nonlinear structure
Linear PCA cannot unfold concentric circles; RBF kernel PCA can.

In [ ]:
Xc2, yc2 = make_circles(n_samples=400, factor=0.3, noise=0.05, random_state=0)
Z_lin = PCA(2).fit_transform(Xc2)
Z_k = KernelPCA(n_components=2, kernel="rbf", gamma=10).fit_transform(Xc2)
fig, axes = plt.subplots(1, 3, figsize=(15, 4.5))
axes[0].scatter(Xc2[:, 0], Xc2[:, 1], c=yc2, cmap="coolwarm", s=10); axes[0].set_title("original")
axes[1].scatter(Z_lin[:, 0], Z_lin[:, 1], c=yc2, cmap="coolwarm", s=10); axes[1].set_title("linear PCA")
axes[2].scatter(Z_k[:, 0], Z_k[:, 1], c=yc2, cmap="coolwarm", s=10); axes[2].set_title("RBF kernel PCA")
plt.show()

### 7.1 Kernel PCA from scratch
1. Build $\mathbf K$. 2. Centre it: $\tilde{\mathbf K}=\mathbf K-\mathbf 1\mathbf K-\mathbf K\mathbf 1+\mathbf 1\mathbf K\mathbf 1$ with $\mathbf 1 = \frac1n\mathbf{11}^\top$. 3. Eigen-decompose. 4. Projections are $\sqrt{\lambda_j}\,\boldsymbol\alpha_j$.

In [ ]:
from sklearn.metrics.pairwise import rbf_kernel
def kernel_pca(X, k=2, gamma=10):
    n = len(X)
    K = rbf_kernel(X, gamma=gamma)
    one = np.ones((n, n)) / n
    Kc = K - one @ K - K @ one + one @ K @ one
    vals, vecs = np.linalg.eigh(Kc)
    vals, vecs = vals[::-1][:k], vecs[:, ::-1][:, :k]
    return vecs * np.sqrt(np.maximum(vals, 0))

Z_mine = kernel_pca(Xc2)
plt.scatter(Z_mine[:, 0], Z_mine[:, 1], c=yc2, cmap="coolwarm", s=10)
plt.title("Kernel PCA (from scratch)"); plt.show()

## 8. Exercises
1. Prove numerically that the PCA scores are **uncorrelated**: compute the covariance matrix of `full.transform(X)`.
2. Implement **LDA** from scratch (scatter matrices $\mathbf S_W$, $\mathbf S_B$ and the generalised eigenproblem via `scipy.linalg.eigh`) and compare with scikit-learn.
3. Apply PCA **without standardisation** to the breast cancer dataset. Which features dominate PC1 and why?
4. Compare `PCA(svd_solver="randomized")` and full PCA in time and accuracy on a random $5000 \times 1000$ matrix.
5. (Optional) Install `umap-learn` in Colab (`!pip install umap-learn`) and compare UMAP with t-SNE on digits.

In [ ]:
# Exercise 1 (starter)
Z = full.transform(X)
C = np.cov(Z, rowvar=False)
print("max |off-diagonal covariance|:", np.abs(C - np.diag(np.diag(C))).max())